# 04 - E-commerce / Customer Support Bot Fine-Tuning\n\nNotebook-first pipeline for local LoRA fine-tuning using subset samples from:\n- Bitext Customer Support\n- Amazon QA\n\nThis notebook is intentionally optimized for fast local iteration.\n

In [ ]:
# Install once\n# !pip install -U pip\n# !pip install -e .\n

In [ ]:
# Hardware check\n!nvidia-smi\n

In [ ]:
from pathlib import Path\nimport yaml\n\nconfig = {\n    'project_name': 'ecommerce_customer_support_bot',\n    'seed': 42,\n    'model': {\n        'base_model_name': 'EleutherAI/gpt-neo-125M',\n        'max_seq_length': 512,\n        'load_in_4bit': False,\n        'torch_dtype': 'float16',\n        'lora_r': 16,\n        'lora_alpha': 32,\n        'lora_dropout': 0.05,\n        'target_modules': ['q_proj', 'v_proj']\n    },\n    'data': {\n        'bitext_dataset_candidates': ['bitext/Bitext-customer-support-llm-chatbot-training-dataset'],\n        'amazon_qa_dataset_candidates': ['mteb/amazon_qa', 'donfu/oa-amazon-qa'],\n        'cache_dir': 'data/cache',\n        'processed_dir': 'data/processed',\n        'bitext_scan_examples': 120000,\n        'amazon_scan_examples': 200000,\n        'train_split': 0.9,\n        'val_split': 0.05,\n        'test_split': 0.05,\n        'max_samples_total': 14000,\n        'max_samples_per_dataset': {\n            'bitext_customer_support': 7000,\n            'amazon_qa': 7000\n        },\n        'min_prompt_chars': 8,\n        'min_response_chars': 8\n    },\n    'training': {\n        'output_dir': 'models/ecommerce_support',\n        'per_device_train_batch_size': 4,\n        'per_device_eval_batch_size': 4,\n        'gradient_accumulation_steps': 8,\n        'num_train_epochs': 1,\n        'learning_rate': 2e-4,\n        'weight_decay': 0.01,\n        'warmup_ratio': 0.03,\n        'logging_steps': 10,\n        'save_strategy': 'epoch',\n        'evaluation_strategy': 'epoch',\n        'gradient_checkpointing': True,\n        'fp16': True,\n        'max_grad_norm': 0.3\n    },\n    'generation': {\n        'max_new_tokens': 180,\n        'temperature': 0.7,\n        'top_p': 0.9,\n        'repetition_penalty': 1.05\n    }\n}\n\nconfig_path = Path('../configs/train_config.yaml').resolve()\nwith open(config_path, 'w', encoding='utf-8') as f:\n    yaml.safe_dump(config, f, sort_keys=False)\n\nprint(config_path)\n

In [ ]:
# Build and cache cleaned/tokenized subsets\n!python ../scripts/prepare_data.py --config configs/train_config.yaml\n

In [ ]:
# LoRA fine-tuning\n!accelerate launch ../scripts/train.py --config configs/train_config.yaml\n

In [ ]:
# BLEU / ROUGE / exact-match accuracy\n!python ../scripts/evaluate.py --config configs/train_config.yaml --split test --max_eval_samples 300\n

In [ ]:
# Serve with uvicorn\n# !python ../scripts/serve.py --config configs/train_config.yaml --host 0.0.0.0 --port 8000\n